# 00 — Dataset Construction

Builds `data/prompts/prompts.csv` containing all conflict and unambiguous prompt pairs.

| Category | Source | N conflict | N unambiguous |
|---|---|---|---|
| A | Winograd Schema Challenge (WCS273) | 20–30 | 20–30 |
| B | Manually authored instruction conflicts | 10 | 10 |
| C | Manually authored factual overrides | 11 | 11 |

**After running this notebook**, all ground-truth labels are filled in and the
CSV is ready for Phases 1–5.

---
**Expected runtime:** ~10 minutes (model inference on WCS273 prompts)

In [ ]:
import sys, os
# Add src/ to path so circuit_conflict can be imported without installing
sys.path.insert(0, os.path.join(os.path.dirname(os.getcwd()), 'src'))

import torch
import pandas as pd
from datasets import load_dataset

from circuit_conflict.utils import load_model, get_device, get_answer_token_id, logit_diff
from circuit_conflict.dataset import (
    build_winograd_rows,
    build_category_b_df,
    build_category_c_df,
    save_prompts,
    PROMPTS_CSV,
)

DEVICE = get_device()
print(f'Device: {DEVICE}')

## 1. Load GPT-2 Small

In [ ]:
model = load_model(DEVICE)

## 2. Category A — Winograd Schema Challenge

Load WCS273 from HuggingFace and filter for **genuinely conflicted** cases:
prompts where the model's margin between the two coreference candidates is
< 15% of the total logit mass on those two tokens.

In [ ]:
print('Loading Winograd WSC273...')
wsc = load_dataset('winograd_wsc', 'wsc273', trust_remote_code=True)
# The dataset has a single 'test' split
wsc_split = wsc['test']
print(f'Total examples: {len(wsc_split)}')
print('Sample:', wsc_split[0])

In [ ]:
print('Filtering for conflicted examples (margin < 15%)...')
df_A = build_winograd_rows(
    wsc_dataset=wsc_split,
    model=model,
    confidence_threshold=0.15,
)
print(f'Category A rows: {len(df_A)}')
print(f'  Conflict rows  : {df_A.is_conflict.sum()}')
print(f'  Unambiguous rows: {(~df_A.is_conflict).sum()}')
df_A.head(4)

### Inspect distribution of margins
A histogram of logit diff margins helps justify the threshold choice.

In [ ]:
import matplotlib.pyplot as plt

# Re-compute margins for all examples to show the full distribution
all_margins = []
for ex in wsc_split:
    span1 = ex['span1_text'].split()[0]
    span2 = ex['span2_text'].split()[0]
    try:
        tok_A = get_answer_token_id(model, span1)
        tok_B = get_answer_token_id(model, span2)
    except ValueError:
        continue
    tokens = model.to_tokens(ex['text'])
    with torch.no_grad():
        logits = model(tokens)
    diff = logit_diff(logits, tok_A, tok_B)
    total = abs(logits[0,-1,tok_A].item()) + abs(logits[0,-1,tok_B].item())
    all_margins.append(abs(diff) / (total + 1e-8))

plt.figure(figsize=(8, 4))
plt.hist(all_margins, bins=30, edgecolor='black', alpha=0.8)
plt.axvline(0.15, color='red', linestyle='--', label='threshold=0.15')
plt.xlabel('Relative margin between competing answers')
plt.ylabel('Count')
plt.title('WCS273: Distribution of GPT-2 confidence margins')
plt.legend()
plt.tight_layout()
plt.savefig('../figures/00_winograd_margin_distribution.png', dpi=150)
plt.show()
print(f'Examples with margin < 0.15: {sum(m < 0.15 for m in all_margins)} / {len(all_margins)}')

## 3. Category B — Instruction Conflict

In [ ]:
df_B = build_category_b_df()
print(f'Category B rows (before ground-truth labelling): {len(df_B)}')
df_B.head(4)

In [ ]:
# Fill in ground_truth for conflict rows by running the model
for idx, row in df_B[df_B.is_conflict].iterrows():
    try:
        tok_A = get_answer_token_id(model, row['answer_A'])
        tok_B = get_answer_token_id(model, row['answer_B'])
    except ValueError as e:
        print(f'  Skipping {row.prompt_id}: {e}')
        continue
    tokens = model.to_tokens(row['prompt_text'])
    with torch.no_grad():
        logits = model(tokens)
    diff = logit_diff(logits, tok_A, tok_B)
    gt = 'A' if diff > 0 else 'B'
    df_B.at[idx, 'ground_truth'] = gt
    print(f'  {row.prompt_id}: logit_diff={diff:.3f} → {gt}')

print('\nGround truth distribution:')
print(df_B[df_B.is_conflict]['ground_truth'].value_counts())

## 4. Category C — Factual vs. Contextual Override

In [ ]:
df_C = build_category_c_df()
print(f'Category C rows (before ground-truth labelling): {len(df_C)}')

In [ ]:
# Fill in ground_truth for conflict rows
for idx, row in df_C[df_C.is_conflict].iterrows():
    try:
        tok_A = get_answer_token_id(model, row['answer_A'])
        tok_B = get_answer_token_id(model, row['answer_B'])
    except ValueError as e:
        print(f'  Skipping {row.prompt_id}: {e}')
        continue
    tokens = model.to_tokens(row['prompt_text'])
    with torch.no_grad():
        logits = model(tokens)
    diff = logit_diff(logits, tok_A, tok_B)
    gt = 'A' if diff > 0 else 'B'
    df_C.at[idx, 'ground_truth'] = gt
    print(f'  {row.prompt_id}: logit_diff={diff:.3f} → {gt} ({"context" if gt=="A" else "trained knowledge"})')

print('\nGround truth distribution (A=context wins, B=trained knowledge wins):')
print(df_C[df_C.is_conflict]['ground_truth'].value_counts())

## 5. Merge all categories and save

In [ ]:
df_all = pd.concat([df_A, df_B, df_C], ignore_index=True)
print(f'Total prompts: {len(df_all)}')
print('\nBreakdown:')
print(df_all.groupby(['category', 'is_conflict']).size().rename('count'))

save_prompts(df_all)
df_all.sample(5, random_state=42)

## 6. Sanity checks

In [ ]:
# Verify every row has a valid ground_truth label
bad = df_all[~df_all['ground_truth'].isin(['A', 'B'])]
if len(bad) > 0:
    print(f'WARNING: {len(bad)} rows with missing ground_truth:')
    print(bad[['prompt_id', 'ground_truth']])
else:
    print('✓ All rows have valid ground_truth labels')

# Count pairs
n_conflict = df_all.is_conflict.sum()
n_unamb = (~df_all.is_conflict).sum()
print(f'✓ {n_conflict} conflict prompts, {n_unamb} unambiguous prompts')
assert n_conflict == n_unamb, 'Mismatch — every conflict should have a paired unambiguous version'
print('✓ Conflict / unambiguous counts match')
print(f'\nDataset saved to: {PROMPTS_CSV}')